In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor


In [2]:
df = pd.read_csv('../data/cleaned_df.csv')
df = df.drop_duplicates()
df

/var/folders/5g/sd7vmfvs2rn86tg601yfsjx80000gn/T/ipykernel_33845/2579964420.py:1: DtypeWarning: Columns (0: PoolPrivateYN, 1: FireplaceYN) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/cleaned_df.csv')


,ClosePrice,LivingArea,LotSizeSquareFeet,BathroomsTotalInteger,Stories,MainLevelBedrooms,BedroomsTotal,DaysOnMarket,Flooring,UnparsedAddress,...,ListingContractDate,StateOrProvince,ViewYN,PoolPrivateYN,AttachedGarageYN,FireplaceYN,NewConstructionYN,ParkingTotal,Longitude,Latitude
0,1650000.0,3452.0,11255.0,4.0,2.0,2.0,5.0,19,NaN,24059 Regents Park Circle,...,2025-02-09,CA,True,True,True,True,False,3.0,-118.558321,34.404533
1,500000.0,2961.0,8258.0,3.0,2.0,1.0,5.0,25,NaN,11592 Greene Court,...,2025-02-11,CA,False,True,True,True,False,2.0,-117.410510,34.522797
2,1650000.0,2929.0,12907.0,3.0,2.0,1.0,5.0,20,"Tile,Wood",2628 Rudy Street,...,2025-03-03,CA,True,False,True,True,False,3.0,-117.866868,33.968692
3,770000.0,1532.0,3300.0,2.0,2.0,2.0,3.0,17,NaN,10602 Porto Court,...,2024-12-27,CA,False,False,False,True,NaN,2.0,-117.102586,32.825896
4,885000.0,2130.0,8100.0,3.0,1.0,4.0,4.0,8,"Carpet,Vinyl",10420 Oneida Avenue,...,2025-03-03,CA,False,False,False,False,False,2.0,-118.426100,34.260374
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80497,594900.0,1867.0,10032.0,3.0,1.0,0.0,4.0,63,NaN,12995 Leith Way,...,2026-02-20,CA,True,True,True,True,False,2.0,-117.036902,34.019226
80498,1705000.0,2776.0,6000.0,3.0,2.0,1.0,4.0,34,"Carpet,Wood",7741 E Briarwood Road,...,2026-04-06,CA,False,True,True,True,False,3.0,-117.770601,33.788646
80499,741000.0,1404.0,6408.0,2.0,1.0,3.0,3.0,4,NaN,2620 W Olive Avenue,...,2026-04-30,CA,False,False,False,False,False,2.0,-117.973674,33.862109
80500,985000.0,1748.0,7310.0,2.0,1.0,3.0,3.0,122,NaN,10505 Halbrent,...,2025-12-09,CA,True,True,False,True,False,2.0,-118.466216,34.261109


In [3]:
target = 'ClosePrice'

# Categorize columns

cat_col = ['Flooring', 'UnparsedAddress', 'AssociationFeeFrequency', 
        'MLSAreaMajor', 'ElementarySchool', 'SubdivisionName', 'City', 
        'PurchaseContractDate', 'MiddleOrJuniorSchool', 'HighSchool',
        'HighSchoolDistrict', 'Levels', 'ListingKey', 'CloseDate', 
        'PropertyType', 'ListingKeyNumeric', 'CountyOrParish',
        'PropertySubType', 'ListingId', 'ContractStatusChangeDate',
        'ListingContractDate', 'StateOrProvince']

bool_col = ['ViewYN', 'PoolPrivateYN', 'AttachedGarageYN', 'FireplaceYN', 'NewConstructionYN']

num_col = ['LivingArea', 'LotSizeSquareFeet', 'BathroomsTotalInteger', 
            'Stories', 'MainLevelBedrooms', 'BedroomsTotal', 'DaysOnMarket']

required_cols = ['ParkingTotal', 'Longitude', 'Latitude']


# Keep only columns that exist in the dataframe
num_col = [col for col in num_col if col in df.columns]
cat_col = [col for col in cat_col if col in df.columns]
bool_col = [col for col in bool_col if col in df.columns]

keep_cols = [target] + num_col + cat_col + bool_col + required_cols
df = df[keep_cols].copy()
df = df.drop_duplicates()

df.head()

,ClosePrice,LivingArea,LotSizeSquareFeet,BathroomsTotalInteger,Stories,MainLevelBedrooms,BedroomsTotal,DaysOnMarket,Flooring,UnparsedAddress,...,ListingContractDate,StateOrProvince,ViewYN,PoolPrivateYN,AttachedGarageYN,FireplaceYN,NewConstructionYN,ParkingTotal,Longitude,Latitude
0,1650000.0,3452.0,11255.0,4.0,2.0,2.0,5.0,19,NaN,24059 Regents Park Circle,...,2025-02-09,CA,True,True,True,True,False,3.0,-118.558321,34.404533
1,500000.0,2961.0,8258.0,3.0,2.0,1.0,5.0,25,NaN,11592 Greene Court,...,2025-02-11,CA,False,True,True,True,False,2.0,-117.410510,34.522797
2,1650000.0,2929.0,12907.0,3.0,2.0,1.0,5.0,20,"Tile,Wood",2628 Rudy Street,...,2025-03-03,CA,True,False,True,True,False,3.0,-117.866868,33.968692
3,770000.0,1532.0,3300.0,2.0,2.0,2.0,3.0,17,NaN,10602 Porto Court,...,2024-12-27,CA,False,False,False,True,NaN,2.0,-117.102586,32.825896
4,885000.0,2130.0,8100.0,3.0,1.0,4.0,4.0,8,"Carpet,Vinyl",10420 Oneida Avenue,...,2025-03-03,CA,False,False,False,False,False,2.0,-118.426100,34.260374


In [4]:

def preproc_df(train_df, test_df):
    train_df = train_df.copy()
    test_df = test_df.copy()



    # ----------------------------
    # Drop rows with missing values in key columns
    # ----------------------------
    required_cols = ['ParkingTotal', 'Longitude', 'Latitude']
    train_df = train_df.dropna(subset=required_cols)
    test_df = test_df.dropna(subset=required_cols)

    # Remove invalid values
    train_df = train_df[
        (train_df["ClosePrice"] > 0) &
        (train_df["LivingArea"] > 0) &
        (train_df["BathroomsTotalInteger"] > 0) &
        (train_df["DaysOnMarket"] > 0)
    ]

    test_df = test_df[
        (test_df["ClosePrice"] > 0) &
        (test_df["LivingArea"] > 0) &
        (test_df["BathroomsTotalInteger"] > 0) &
        (test_df["DaysOnMarket"] > 0)
    ]

    # ----------------------------
    # Missing value handling
    # ----------------------------

    for col in cat_col:
        if col in train_df.columns:
            train_df[col] = train_df[col].fillna("Unknown")
        if col in test_df.columns:
            test_df[col] = test_df[col].fillna("Unknown")

    # Missing indicator creation
    for col in train_df.columns:
        if train_df[col].isna().sum() > 0:
            train_df[f"{col}_was_missing"] = train_df[col].isna().astype(int) #
            test_df[f"{col}_was_missing"] = test_df[col].isna().astype(int) #

    if "YearBuilt" in train_df.columns:
        median = train_df["YearBuilt"].median()
        train_df["YearBuilt"] = train_df["YearBuilt"].fillna(median)
        test_df["YearBuilt"] = test_df["YearBuilt"].fillna(median)

    if "StreetNumberNumeric" in train_df.columns:
        train_df["StreetNumberNumeric"] = train_df["StreetNumberNumeric"].fillna(-1)
        test_df["StreetNumberNumeric"] = test_df["StreetNumberNumeric"].fillna(-1)

    train_df[bool_col] = train_df[bool_col].fillna(False)
    test_df[bool_col] = test_df[bool_col].fillna(False)

    for col in num_col:
        if col in train_df.columns:
            median = train_df[col].median()
            train_df[col] = train_df[col].fillna(median)
            test_df[col] = test_df[col].fillna(median)

    if "AssociationFee" in train_df.columns:
        train_df["AssociationFee"] = train_df["AssociationFee"].fillna(0)
        test_df["AssociationFee"] = test_df["AssociationFee"].fillna(0)

    for col in ["GarageSpaces", "ParkingTotal"]:
        if col in train_df.columns:
            train_df[col] = train_df[col].fillna(0)
            test_df[col] = test_df[col].fillna(0)

    # ----------------------------
    # Boolean encoding
    # ----------------------------

    train_df[bool_col] = train_df[bool_col].astype(int)
    test_df[bool_col] = test_df[bool_col].astype(int)

    # ----------------------------
    # MultiLabel Encoding
    # ----------------------------

    multi_cols = ["Flooring", "Levels"]

    for col in multi_cols:
        train_df[col] = train_df[col].fillna("").str.split(",")
        test_df[col] = test_df[col].fillna("").str.split(",")

        mlb = MultiLabelBinarizer()

        train_encoded = pd.DataFrame(
            mlb.fit_transform(train_df[col]),
            columns=[f"{col}_{c}" for c in mlb.classes_],
            index=train_df.index
        )

        test_encoded = pd.DataFrame(
            mlb.transform(test_df[col]),
            columns=[f"{col}_{c}" for c in mlb.classes_],
            index=test_df.index
        )

        train_df = train_df.drop(columns=col).join(train_encoded)
        test_df = test_df.drop(columns=col).join(test_encoded)

    # ----------------------------
    # Ordinal Encoding
    # ----------------------------

    mapping = {
        "Unknown": 0,
        "Monthly": 1,
        "Quarterly": 2,
        "SemiAnnually": 3,
        "Annually": 4
    }

    train_df["AssociationFeeFrequency"] = train_df["AssociationFeeFrequency"].map(mapping)
    test_df["AssociationFeeFrequency"] = test_df["AssociationFeeFrequency"].map(mapping)

    # ----------------------------
    # One-Hot Encoding
    # ----------------------------

    one_hot = [
        "CountyOrParish",
        "StateOrProvince",
        "City",
        "PropertyType",
        "PropertySubType",
    ]

    for col in one_hot:
        top = train_df[col].value_counts().head(200).index

        train_df[col] = train_df[col].where(train_df[col].isin(top), "Other")
        test_df[col] = test_df[col].where(test_df[col].isin(top), "Other")

    train_df = pd.get_dummies(train_df, columns=one_hot, drop_first=True)
    test_df = pd.get_dummies(test_df, columns=one_hot, drop_first=True)

    # Ensure same columns
    train_df, test_df = train_df.align(test_df, join="left", axis=1, fill_value=0)

    # ----------------------------
    # Standardization
    # ----------------------------

    scale = [
        "LivingArea",
        "LotSizeSquareFeet",
        "AssociationFee",
        "DaysOnMarket"
    ]

    scale = [c for c in scale if c in train_df.columns]

    scaler = StandardScaler()
    train_df[scale] = scaler.fit_transform(train_df[scale])
    test_df[scale] = scaler.transform(test_df[scale])

    train_df = train_df.drop(columns=cat_col, errors='ignore')
    test_df = test_df.drop(columns=cat_col, errors='ignore')

    train_df = train_df.drop_duplicates()
    test_df = test_df.drop_duplicates()


    return train_df, test_df

In [9]:
# # Convert CloseDate to datetime
df['CloseDate'] = pd.to_datetime(df['CloseDate'])
df = df.sort_values('CloseDate').reset_index(drop=True)

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(max_depth=5),
    "Random Forest": RandomForestRegressor(n_estimators=100)
}
model_results = []

# # Split dataframe into train and test sets using given window
def get_time_split(data, X_months):
    latest_date = data['CloseDate'].max()
    test_start_date = latest_date - pd.DateOffset(months=1)
    train_start_date = test_start_date - pd.DateOffset(months=X_months)
    test_set = data[data['CloseDate'] >= test_start_date]
    train_set = data[(data['CloseDate'] >= train_start_date) & (data['CloseDate'] < test_start_date)]
    return train_set, test_set

window_options = [3, 6, 9, 12]
for X in window_options:
    train, test = get_time_split(df, X_months=X)
    train_df, test_df = preproc_df(train, test)

    X_train = train_df.drop(columns=['ClosePrice', 'DaysOnMarket'])
    y_train = train_df[target]
    X_test = test_df.drop(columns=['ClosePrice', 'DaysOnMarket'])
    y_test = test_df[target]

    for model_name, model in models.items():

        model.fit(X_train, y_train)

        train_preds = model.predict(X_train)
        test_preds = model.predict(X_test)

        train_r2 = r2_score(y_train, train_preds)
        test_r2 = r2_score(y_test, test_preds)

        mae = mean_absolute_error(y_test, test_preds)
        rmse = np.sqrt(mean_squared_error(y_test,test_preds))

        model_results.append({
            "model": model_name,
            "month_split": X,
            "train_r2": train_r2,
            "test_r2": test_r2,
            "mae": mae,
            "rmse": rmse
        })

comparison_df = pd.DataFrame(model_results)

comparison_df.sort_values(
    by="test_r2",
    ascending=False
)


,model,month_split,train_r2,test_r2,mae,rmse
11,Random Forest,12,0.986952,0.911914,98273.960602,173313.554705
8,Random Forest,9,0.986019,0.905206,101393.058474,179791.331518
5,Random Forest,6,0.984444,0.898299,105232.115703,186226.399341
2,Random Forest,3,0.982806,0.885712,111512.451210,197414.855503
1,Decision Tree,3,0.674696,0.661602,226761.327998,339697.513239
10,Decision Tree,12,0.657183,0.654050,231739.337753,343467.490214
7,Decision Tree,9,0.656010,0.652239,230551.933630,344365.208387
4,Decision Tree,6,0.658932,0.651182,231703.245661,344887.903110
9,Linear Regression,12,0.800537,0.305265,424075.609780,486730.084809
6,Linear Regression,9,0.801054,0.284876,431431.153902,493820.909169


#### Models
> The R2 scores for the **Linear Regression** models are the smallest meaning the data likely doesn't have strong linear relationships. Moving forward with a linear model would mean performing transformations and feature engineering to help the model perform better

> The R2 scores for the **Decision Tree** models were better with a similar training and testing scores. This means the model isn't overfitting but it also isn't capturing much of the pattern in the data, which is to be expected for a baseline model

> The R2 scores for the **Random Forest** models were the best of all the models and is likely the strongest model to build on with this data
